In [1]:
import pandas as pd

In [4]:
import os

folder_path = r"C:\Users\shari\OneDrive\Desktop\Projects\Cyclistic bike\Original files"

if os.path.exists(folder_path):
    print("The folder exists")
    
    files = os.listdir(folder_path)
    print(f"\nNumber of files in the folder: {len(files)}")
    print("\n files:")
    for f in files[:13]:
        print(f"  - {f}")
else:
    print("The folder does not exist")

The folder exists

Number of files in the folder: 14

 files:
  - .ipynb_checkpoints
  - 202501-divvy-tripdata.csv
  - 202502-divvy-tripdata.csv
  - 202503-divvy-tripdata.csv
  - 202504-divvy-tripdata.csv
  - 202505-divvy-tripdata.csv
  - 202506-divvy-tripdata.csv
  - 202507-divvy-tripdata.csv
  - 202508-divvy-tripdata.csv
  - 202509-divvy-tripdata.csv
  - 202510-divvy-tripdata.csv
  - 202511-divvy-tripdata.csv
  - 202512-divvy-tripdata.csv


In [6]:
folder_path = r"C:\Users\shari\OneDrive\Desktop\Projects\Cyclistic bike\Original files"
all_data = []
for file in os.listdir(folder_path):
    if file.endswith('.csv') and file.startswith('2025'):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        all_data.append(df)
        print(f"Loaded {file} - number of rows: {len(df):,}")
df_2025 = pd.concat(all_data, ignore_index=True)

print(f"\nTotal number of trips in 2025: {len(df_2025):,}")


Loaded 202501-divvy-tripdata.csv - number of rows: 138,689
Loaded 202502-divvy-tripdata.csv - number of rows: 151,880
Loaded 202503-divvy-tripdata.csv - number of rows: 298,155
Loaded 202504-divvy-tripdata.csv - number of rows: 371,341
Loaded 202505-divvy-tripdata.csv - number of rows: 502,456
Loaded 202506-divvy-tripdata.csv - number of rows: 678,904
Loaded 202507-divvy-tripdata.csv - number of rows: 763,432
Loaded 202508-divvy-tripdata.csv - number of rows: 790,177
Loaded 202509-divvy-tripdata.csv - number of rows: 714,759
Loaded 202510-divvy-tripdata.csv - number of rows: 646,039
Loaded 202511-divvy-tripdata.csv - number of rows: 356,628
Loaded 202512-divvy-tripdata.csv - number of rows: 140,534

Total number of trips in 2025: 5,552,994


In [7]:
df_2025['started_at'] = pd.to_datetime(df_2025['started_at'])
df_2025['ended_at'] = pd.to_datetime(df_2025['ended_at'])

df_2025['ride_length'] = (df_2025['ended_at'] - df_2025['started_at']).dt.total_seconds() / 60

df_2025['day_of_week'] = df_2025['started_at'].dt.day_name()
df_2025['month'] = df_2025['started_at'].dt.month

print("New columns created")
print(f"\nFirst 3 rows of the new columns:")
print(df_2025[['ride_length', 'day_of_week', 'month']].head(3))

New columns created

First 3 rows of the new columns:
   ride_length day_of_week  month
0    13.957950     Tuesday      1
1     5.072400    Saturday      1
2    11.591667    Thursday      1


In [8]:
# Save the number of trips before cleaning
before_clean = len(df_2025)

# Remove trips shorter than 1 minute or longer than 24 hours (1440 minutes)
df_2025_clean = df_2025[(df_2025['ride_length'] >= 1) & (df_2025['ride_length'] <= 1440)]

# Calculate the number of trips removed
after_clean = len(df_2025_clean)
removed = before_clean - after_clean

print(f"Before cleaning: {before_clean:,} trips")
print(f"Removed: {removed:,} trips (shorter than 1 minute or longer than 24 hours)")
print(f"After cleaning: {after_clean:,} trips")
print(f"\nPercentage of remaining data: {(after_clean/before_clean)*100:.2f}%")

Before cleaning: 5,552,994 trips
Removed: 152,986 trips (shorter than 1 minute or longer than 24 hours)
After cleaning: 5,400,008 trips

Percentage of remaining data: 97.24%


In [9]:
# Comparison between user types
stats = df_2025_clean.groupby('member_casual')['ride_length'].agg(['mean', 'median', 'count', 'min', 'max'])
stats = stats.rename(columns={
    'mean': 'Average duration (minutes)',
    'median': 'Median (minutes)', 
    'count': 'Number of trips',
    'min': 'Shortest trip (minutes)',
    'max': 'Longest trip (minutes)'
})

print("Comparison between annual members and casual users:")
print(stats)

Comparison between annual members and casual users:
               Average duration (minutes)  Median (minutes)  Number of trips  \
member_casual                                                                  
casual                          19.906370         11.893208          1915806   
member                          12.179841          8.738008          3484202   

               Shortest trip (minutes)  Longest trip (minutes)  
member_casual                                                   
casual                        1.000017             1439.975950  
member                        1.000000             1439.901683  


In [10]:
# Arrange days of the week in correct order
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Number of trips per day by user type
daily_rides = df_2025_clean.groupby(['member_casual', 'day_of_week']).size().unstack()
daily_rides = daily_rides[days_order]

print("Number of trips by day of the week:")
print(daily_rides)

# Calculate percentage for each day
print("\nPercentage of trips per day:")
daily_percent = daily_rides.div(daily_rides.sum(axis=1), axis=0) * 100
print(daily_percent.round(1))

Number of trips by day of the week:
day_of_week    Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
member_casual                                                                
casual         219232   216926     212744    247893  306463    395595  316953
member         493322   552529     540212    565177  518551    439876  374535

Percentage of trips per day:
day_of_week    Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
member_casual                                                                
casual           11.4     11.3       11.1      12.9    16.0      20.6    16.5
member           14.2     15.9       15.5      16.2    14.9      12.6    10.7


In [11]:
# Number of rides per month by user type
monthly_rides = df_2025_clean.groupby(['member_casual', 'month']).size().unstack()

# Month names for clarity
month_names = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

monthly_rides.columns = [month_names[m] for m in monthly_rides.columns]

print("Number of rides per month:")
print(monthly_rides)

# Calculate percentage for each month
print("\nPercentage of rides per month:")
monthly_percent = monthly_rides.div(monthly_rides.sum(axis=1), axis=0) * 100
print(monthly_percent.round(1))

Number of rides per month:
                  Jan     Feb     Mar     Apr     May     Jun     Jul     Aug  \
member_casual                                                                   
casual          23405   27003   82864  105260  175655  278702  308446  323533   
member         112331  122097  208458  257921  313996  379523  430394  443130   

                  Sep     Oct     Nov     Dec  
member_casual                                  
casual         254727  214380   94719   27112  
member         440954  414094  251924  109380  

Percentage of rides per month:
               Jan  Feb  Mar  Apr  May   Jun   Jul   Aug   Sep   Oct  Nov  Dec
member_casual                                                                 
casual         1.2  1.4  4.3  5.5  9.2  14.5  16.1  16.9  13.3  11.2  4.9  1.4
member         3.2  3.5  6.0  7.4  9.0  10.9  12.4  12.7  12.7  11.9  7.2  3.1


In [12]:
# Average ride duration per month by user type
monthly_avg_duration = df_2025_clean.groupby(['member_casual', 'month'])['ride_length'].mean().unstack()
monthly_avg_duration.columns = [month_names[m] for m in monthly_avg_duration.columns]

print("Average ride duration (minutes) per month:")
print(monthly_avg_duration.round(1))


Average ride duration (minutes) per month:
                Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct  \
member_casual                                                               
casual         12.0  12.4  17.9  18.7  20.8  21.9  21.4  21.7  19.6  18.1   
member         10.0  10.0  11.1  11.3  11.9  12.8  13.2  13.1  12.6  12.1   

                Nov   Dec  
member_casual              
casual         15.5  12.9  
member         11.7  11.7  


In [13]:
# Extract start hour
df_2025_clean['start_hour'] = df_2025_clean['started_at'].dt.hour

# Number of rides by hour and user type
hourly_rides = df_2025_clean.groupby(['member_casual', 'start_hour']).size().unstack()

print("Number of rides by hour of the day:")
print(hourly_rides)

# Percentage for each hour
print("\nPercentage of rides by hour of the day:")
hourly_percent = hourly_rides.div(hourly_rides.sum(axis=1), axis=0) * 100
print(hourly_percent.round(1))

C:\Users\shari\AppData\Local\Temp\ipykernel_3140\68123133.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025_clean['start_hour'] = df_2025_clean['started_at'].dt.hour


Number of rides by hour of the day:
start_hour        0      1      2     3     4      5      6       7       8   \
member_casual                                                                  
casual         36762  23492  15667  8694  6941  11009  25696   47925   68010   
member         31391  19107  11430  7520  8606  33525  99153  196413  252034   

start_hour         9   ...      14      15      16      17      18      19  \
member_casual          ...                                                   
casual          68273  ...  133239  148055  168789  183262  156921  116407   
member         163882  ...  184831  232972  328673  375149  290116  200336   

start_hour         20      21     22     23  
member_casual                                
casual          83803   71204  59981  43469  
member         138362  107972  78612  48926  

[2 rows x 24 columns]

Percentage of rides by hour of the day:
start_hour      0    1    2    3    4    5    6    7    8    9   ...   14  \
membe

In [14]:
# Compare bike types
bike_type = df_2025_clean.groupby(['member_casual', 'rideable_type']).size().reset_index(name='count')

# Add percentage column
total_by_type = df_2025_clean.groupby('member_casual').size().reset_index(name='total')
bike_type = bike_type.merge(total_by_type, on='member_casual')
bike_type['percentage'] = (bike_type['count'] / bike_type['total']) * 100

print("Distribution of bike types:")
print(bike_type[['member_casual', 'rideable_type', 'count', 'percentage']].round(1))

Distribution of bike types:
  member_casual  rideable_type    count  percentage
0        casual   classic_bike   667993        34.9
1        casual  electric_bike  1247813        65.1
2        member   classic_bike  1274451        36.6
3        member  electric_bike  2209751        63.4


In [15]:
# Top start stations for each user type
top_stations = df_2025_clean.groupby(['member_casual', 'start_station_name']).size().reset_index(name='count')
top_stations = top_stations.sort_values(['member_casual', 'count'], ascending=[True, False])

print("Top 5 stations for casual users:")
print(top_stations[top_stations['member_casual'] == 'casual'].head(5))

print("\nTop 5 stations for annual members:")
print(top_stations[top_stations['member_casual'] == 'member'].head(5))

Top 5 stations for casual users:
     member_casual                  start_station_name  count
322         casual   DuSable Lake Shore Dr & Monroe St  30712
803         casual                           Navy Pier  26939
1540        casual             Streeter Dr & Grand Ave  23288
752         casual               Michigan Ave & Oak St  22058
323         casual  DuSable Lake Shore Dr & North Blvd  19023

Top 5 stations for annual members:
     member_casual            start_station_name  count
2256        member      Kingsbury St & Kinzie St  30889
1926        member  Clinton St & Washington Blvd  25299
1838        member         Canal St & Madison St  21837
1922        member       Clinton St & Madison St  21818
1893        member             Clark St & Elm St  20602


In [16]:
# Top 10 longest rides for each user type
print("Top 10 longest rides - Casual users:")
longest_casual = df_2025_clean[df_2025_clean['member_casual'] == 'casual'].nlargest(10, 'ride_length')
print(longest_casual[['ride_length', 'start_station_name', 'end_station_name', 'day_of_week']])

print("\nTop 10 longest rides - Annual members:")
longest_member = df_2025_clean[df_2025_clean['member_casual'] == 'member'].nlargest(10, 'ride_length')
print(longest_member[['ride_length', 'start_station_name', 'end_station_name', 'day_of_week']])

Top 10 longest rides - Casual users:
         ride_length                   start_station_name  \
4952640  1439.975950             Jefferson St & Monroe St   
5372553  1439.955050       Southport Ave & Wrightwood Ave   
5214829  1439.943317             Wallace St & Pershing Rd   
5212171  1439.937983             Halsted St & Dickens Ave   
5371631  1439.931267  DuSable Lake Shore Dr & Belmont Ave   
5099495  1439.924050                Halsted St & 119th St   
5380431  1439.923667                  MLK Jr Dr & 29th St   
5379294  1439.914850      Cityfront Plaza Dr & Pioneer Ct   
5379293  1439.910250      Cityfront Plaza Dr & Pioneer Ct   
5154235  1439.897850               State St & Randolph St   

                           end_station_name day_of_week  
4952640  DuSable Lake Shore Dr & North Blvd      Sunday  
5372553                                 NaN    Saturday  
5214829                                 NaN    Saturday  
5212171                                 NaN      Sunday  
5

In [18]:
# Clean long rides (more than one hour) by removing missing values
long_rides_clean = df_2025_clean[
    (df_2025_clean['ride_length'] > 60) & 
    (df_2025_clean['end_station_name'].notna()) &
    (df_2025_clean['start_station_name'].notna())
]

print(f"Number of long rides (more than one hour) after cleaning: {len(long_rides_clean):,}")
print(f"Out of a total of {len(df_2025_clean):,} rides")
print(f"Percentage: {(len(long_rides_clean)/len(df_2025_clean))*100:.2f}%")

Number of long rides (more than one hour) after cleaning: 95,769
Out of a total of 5,400,008 rides
Percentage: 1.77%


In [19]:
# Long ride statistics by user type
long_rides_stats = long_rides_clean.groupby('member_casual')['ride_length'].agg(['mean', 'median', 'count', 'min', 'max'])
long_rides_stats = long_rides_stats.rename(columns={
    'mean': 'Average duration (minutes)',
    'median': 'Median duration (minutes)',
    'count': 'Number of long rides',
    'min': 'Shortest long ride',
    'max': 'Longest ride'
})

print("Statistics of long rides (more than one hour):")
print(long_rides_stats)

# Percentage of long rides out of total rides for each user type
total_casual = len(df_2025_clean[df_2025_clean['member_casual'] == 'casual'])
total_member = len(df_2025_clean[df_2025_clean['member_casual'] == 'member'])

long_casual = len(long_rides_clean[long_rides_clean['member_casual'] == 'casual'])
long_member = len(long_rides_clean[long_rides_clean['member_casual'] == 'member'])

print("\nPercentage of long rides out of total rides for each user type:")
print(f"Casual: {long_casual/total_casual*100:.2f}% ({long_casual:,} out of {total_casual:,})")
print(f"Member: {long_member/total_member*100:.2f}% ({long_member:,} out of {total_member:,})")

Statistics of long rides (more than one hour):
               Average duration (minutes)  Median duration (minutes)  \
member_casual                                                          
casual                         120.621049                  84.699183   
member                         162.745138                  83.593900   

               Number of long rides  Shortest long ride  Longest ride  
member_casual                                                          
casual                        81172           60.000650   1439.975950  
member                        14597           60.001867   1439.827217  

Percentage of long rides out of total rides for each user type:
Casual: 4.24% (81,172 out of 1,915,806)
Member: 0.42% (14,597 out of 3,484,202)


In [20]:
# Analyze long rides by day of the week
long_rides_days = long_rides_clean.groupby(['member_casual', 'day_of_week']).size().unstack()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
long_rides_days = long_rides_days[days_order]

print("Distribution of long rides by day of the week:")
print(long_rides_days)

# Percentage
print("\nPercentage of long rides by day of the week:")
long_rides_percent = long_rides_days.div(long_rides_days.sum(axis=1), axis=0) * 100
print(long_rides_percent.round(1))

Distribution of long rides by day of the week:
day_of_week    Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
member_casual                                                                
casual           9904     7522       5860      7777   12006     20075   18028
member           1926     1816       1652      1861    2197      2569    2576

Percentage of long rides by day of the week:
day_of_week    Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
member_casual                                                                
casual           12.2      9.3        7.2       9.6    14.8      24.7    22.2
member           13.2     12.4       11.3      12.7    15.1      17.6    17.6


In [21]:
# Top start stations for long rides (Casual users)
top_long_start_casual = long_rides_clean[long_rides_clean['member_casual'] == 'casual']\
    .groupby('start_station_name').size().reset_index(name='count')\
    .sort_values('count', ascending=False).head(10)

print("Top 10 start stations for long rides - Casual:")
print(top_long_start_casual)

# Top start stations for long rides (Members)
top_long_start_member = long_rides_clean[long_rides_clean['member_casual'] == 'member']\
    .groupby('start_station_name').size().reset_index(name='count')\
    .sort_values('count', ascending=False).head(10)

print("\nTop 10 start stations for long rides - Member:")
print(top_long_start_member)

Top 10 start stations for long rides - Casual:
                     start_station_name  count
299   DuSable Lake Shore Dr & Monroe St   3306
710                           Navy Pier   2862
666               Michigan Ave & Oak St   2852
961             Streeter Dr & Grand Ave   2794
673                     Millennium Park   2387
303                      Dusable Harbor   1550
300  DuSable Lake Shore Dr & North Blvd   1293
661               Michigan Ave & 8th St   1155
966                 Theater on the Lake   1086
668        Michigan Ave & Washington St   1063

Top 10 start stations for long rides - Member:
                     start_station_name  count
259   DuSable Lake Shore Dr & Monroe St    188
590                New St & Illinois St    137
552               Michigan Ave & Oak St    137
759             Sheridan Rd & Argyle St    127
825                 Theater on the Lake    125
589                           Navy Pier    124
260  DuSable Lake Shore Dr & North Blvd    121
263         

In [22]:
# Most popular routes (Start → End) for long rides
popular_routes = long_rides_clean.groupby(['member_casual', 'start_station_name', 'end_station_name'])\
    .size().reset_index(name='count')\
    .sort_values(['member_casual', 'count'], ascending=[True, False])

print("Top 5 routes for long rides - Casual:")
print(popular_routes[popular_routes['member_casual'] == 'casual'].head(5))

print("\nTop 5 routes for long rides - Member:")
print(popular_routes[popular_routes['member_casual'] == 'member'].head(5))

Top 5 routes for long rides - Casual:
      member_casual                 start_station_name  \
14128        casual              Michigan Ave & Oak St   
15647        casual                          Navy Pier   
6898         casual  DuSable Lake Shore Dr & Monroe St   
20184        casual            Streeter Dr & Grand Ave   
15253        casual                    Montrose Harbor   

                        end_station_name  count  
14128              Michigan Ave & Oak St   1102  
15647                          Navy Pier   1049  
6898   DuSable Lake Shore Dr & Monroe St    998  
20184            Streeter Dr & Grand Ave    787  
15253                    Montrose Harbor    527  

Top 5 routes for long rides - Member:
      member_casual          start_station_name            end_station_name  \
30465        member     Sheridan Rd & Argyle St     Sheridan Rd & Argyle St   
24660        member   Clark St & Ida B Wells Dr   Clark St & Ida B Wells Dr   
28488        member      Marine Dr & 

In [24]:
import pandas as pd
import os

# Define base path
base_path = r"C:\Users\shari\OneDrive\Desktop\Projects\Cyclistic bike"

# Confirm df_2025_clean exists
print(f"Base data: {len(df_2025_clean):,} rides, {len(df_2025_clean.columns)} columns")
print(f"Available columns: {df_2025_clean.columns.tolist()}")

# ========================================
# Create additional columns useful for analysis
# ========================================

# Season column
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_2025_clean['season'] = df_2025_clean['month'].apply(get_season)

# Weekend column
df_2025_clean['is_weekend'] = df_2025_clean['day_of_week'].isin(['Saturday', 'Sunday'])

print("Additional columns created: season, is_weekend")

# ========================================
# 1. Main rides file
# ========================================
rides = df_2025_clean[['ride_id', 'member_casual', 'rideable_type']].copy()
rides.to_csv(os.path.join(base_path, '01_rides.csv'), index=False)
print(f"\n01_rides.csv - {len(rides):,} rows, {len(rides.columns)} columns")

# ========================================
# 2. Time file (all time-related columns)
# ========================================
time_cols = ['ride_id', 'started_at', 'ended_at', 'start_hour', 'day_of_week', 'month', 'season', 'is_weekend']
time_df = df_2025_clean[time_cols].copy()
time_df.to_csv(os.path.join(base_path, '02_time.csv'), index=False)
print(f"02_time.csv - {len(time_df):,} rows, {len(time_df.columns)} columns")

# ========================================
# 3. Duration file
# ========================================
duration = df_2025_clean[['ride_id', 'ride_length']].copy()
duration.to_csv(os.path.join(base_path, '03_duration.csv'), index=False)
print(f"03_duration.csv - {len(duration):,} rows, {len(duration.columns)} columns")

# ========================================
# 4. Start stations file
# ========================================
start_stations = df_2025_clean[['ride_id', 'start_station_name', 'start_lat', 'start_lng']].copy()
start_stations.to_csv(os.path.join(base_path, '04_start_stations.csv'), index=False)
print(f"04_start_stations.csv - {len(start_stations):,} rows, {len(start_stations.columns)} columns")

# ========================================
# 5. End stations file
# ========================================
end_stations = df_2025_clean[['ride_id', 'end_station_name', 'end_lat', 'end_lng']].copy()
end_stations.to_csv(os.path.join(base_path, '05_end_stations.csv'), index=False)
print(f"05_end_stations.csv - {len(end_stations):,} rows, {len(end_stations.columns)} columns")

# ========================================
# 6. Summary statistics file (for quick KPIs)
# ========================================
summary = df_2025_clean.groupby('member_casual').agg({
    'ride_length': ['mean', 'median', 'count', 'min', 'max']
}).round(2)
summary.columns = ['avg_minutes', 'median_minutes', 'total_rides', 'min_minutes', 'max_minutes']
summary.reset_index().to_csv(os.path.join(base_path, '06_summary_stats.csv'), index=False)
print(f"06_summary_stats.csv - {len(summary)} rows, {len(summary.columns)} columns")

# ========================================
# Check file sizes
# ========================================
print("\n" + "="*50)
print("File sizes created:")
print("="*50)

file_sizes = []
for i in range(1, 6):
    file_path = os.path.join(base_path, f'0{i}_rides.csv')
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        file_sizes.append(size_mb)
        status = "OK" if size_mb <= 15 else "Warning"
        print(f"{status} 0{i}_rides.csv: {size_mb:.2f} MB")

# Stats file
stats_path = os.path.join(base_path, '06_summary_stats.csv')
if os.path.exists(stats_path):
    size_mb = os.path.getsize(stats_path) / (1024 * 1024)
    print(f"OK 06_summary_stats.csv: {size_mb:.2f} MB")

print("="*50)
print(f"Total space used: {sum(file_sizes):.2f} MB ✅")

Base data: 5,400,008 rides, 17 columns
Available columns: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'ride_length', 'day_of_week', 'month', 'start_hour']


C:\Users\shari\AppData\Local\Temp\ipykernel_3140\2515882505.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025_clean['season'] = df_2025_clean['month'].apply(get_season)
C:\Users\shari\AppData\Local\Temp\ipykernel_3140\2515882505.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025_clean['is_weekend'] = df_2025_clean['day_of_week'].isin(['Saturday', 'Sunday'])


Additional columns created: season, is_weekend

01_rides.csv - 5,400,008 rows, 3 columns
02_time.csv - 5,400,008 rows, 8 columns
03_duration.csv - 5,400,008 rows, 2 columns
04_start_stations.csv - 5,400,008 rows, 4 columns
05_end_stations.csv - 5,400,008 rows, 4 columns
06_summary_stats.csv - 2 rows, 5 columns

File sizes created:
Warning 01_rides.csv: 198.99 MB
OK 06_summary_stats.csv: 0.00 MB
Total space used: 198.99 MB ✅
